# IGRO - notebook para figuras e tabelas do artigo

Este notebook foi pensado para apoiar a elaboracao das tabelas e graficos sugeridos em `10_publicacao/submissao/artigo_igro_v2_critica_aplicada.md` a partir dos dados em `02_dados/processed`.

Escopo:
- ler os arquivos processados, mesmo quando vierem encapsulados em HTML/JavaScript;
- montar as tabelas centrais do artigo;
- gerar os graficos sugeridos nas secoes de resultados;
- acrescentar analises extras uteis para robustez e transparencia metodologica.

Assuncoes operacionais usadas aqui:
- `rdp` foi tratado como `PMA` (percentual de manifestacoes em atraso);
- `tr` foi tratado como `RP` (resolutividade percebida);
- `ri` foi tratado como `%RI`;
- `nps` foi tratado como `NR` em escala NPS;
- as faixas de risco foram inferidas a partir do HTML renderizado do projeto: `Controlado >= 90`, `Atencao >= 70`, `Elevado >= 50`, `Critico < 50`.

Observacao importante: a base processada atualmente traz `52` orgaos, enquanto o artigo menciona `51`. O notebook destaca essa divergencia para revisao editorial/metodologica.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 13

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "02_dados").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz do repositorio.")

ROOT = find_repo_root()
DATA_DIR = ROOT / "02_dados" / "processed"
OUTPUT_DIR = ROOT / "09_resultados" / "artigo_igro_figuras_tabelas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROOT, DATA_DIR, OUTPUT_DIR

In [ ]:
def parse_js_payload(text: str, anchor: str = "const D=") -> list[dict]:
    pattern = rf"{re.escape(anchor)}(\[.*?\]);const"
    match = re.search(pattern, text, flags=re.DOTALL)
    if not match:
        raise ValueError("Payload JavaScript nao encontrado no arquivo.")
    payload = match.group(1)
    payload = re.sub(r"([\{,])(\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1"\3":', payload)
    payload = payload.replace("'", '"')
    return json.loads(payload)

def load_processed_export(path: Path) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8", errors="replace")
    if "const D=" in text:
        return pd.DataFrame(parse_js_payload(text, anchor="const D="))
    return pd.read_csv(path)

def classify_risk(igro_pct: float) -> str:
    if pd.isna(igro_pct):
        return "Sem dado"
    if igro_pct >= 90:
        return "Controlado"
    if igro_pct >= 70:
        return "Atencao"
    if igro_pct >= 50:
        return "Elevado"
    return "Critico"

def save_fig(fig: plt.Figure, name: str) -> None:
    fig.savefig(OUTPUT_DIR / name, dpi=200, bbox_inches="tight")

def save_table(df: pd.DataFrame, name: str) -> None:
    df.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8-sig")

df = load_processed_export(DATA_DIR / "geral.csv")

rename_map = {
    "sigla": "orgao",
    "classe": "classe_operacional",
    "manifestacoes": "manifestacoes",
    "pesquisas": "pesquisas",
    "tmr": "tmr_dias",
    "rdp": "pma",
    "tr": "rp",
    "ri": "ri",
    "nps": "nr_nps",
    "igro": "igro",
    "sub_t": "sub_tempestividade",
    "sub_q": "sub_qualidade",
    "flag_amostra": "flag_amostra"
}
df = df.rename(columns=rename_map)

for col in ["pma", "rp", "ri", "igro", "sub_tempestividade", "sub_qualidade"]:
    df[f"{col}_pct"] = df[col] * 100

df["classe_operacional"] = df["classe_operacional"].astype(str)
df["faixa_risco"] = df["igro_pct"].apply(classify_risk)
df["amostra_insuficiente"] = df["flag_amostra"].astype(int).eq(1)

score_cols = ["score_rdp", "score_tmr", "score_tr", "score_ri", "score_nr"]
score_cols_pct = []
for col in score_cols:
    if col in df.columns:
        df[f"{col}_pct"] = df[col] * 100
        score_cols_pct.append(f"{col}_pct")

df = df.sort_values("igro_pct", ascending=False).reset_index(drop=True)
df.head()

## 1. Checagem inicial da base

Esta secao ajuda a conferir rapidamente se o conjunto usado no artigo bate com o export processado atual.

In [ ]:
resumo_base = pd.DataFrame(
    {
        "metrica": [
            "orgaos na base",
            "manifestacoes totais",
            "pesquisas totais",
            "orgaos com amostra insuficiente",
            "orgaos com IGRO zero",
            "IGRO medio (%)"
        ],
        "valor": [
            len(df),
            int(df["manifestacoes"].sum()),
            int(df["pesquisas"].sum()),
            int(df["amostra_insuficiente"].sum()),
            int(df["igro_pct"].eq(0).sum()),
            round(df["igro_pct"].mean(), 1)
        ]
    }
)
display(resumo_base)

if len(df) != 51:
    print(f"ALERTA: a base processada atual contem {len(df)} orgaos, enquanto o artigo menciona 51.")
    print("Sugestao: revisar se ha algum orgao que deve ser excluido editorialmente antes da versao final.")

## 2. Tabela 4 - distribuicao operacional da rede

Corresponde diretamente a `Tabela 4 - Distribuicao operacional da rede` sugerida no artigo.

In [ ]:
tabela_classes = (
    df.groupby("classe_operacional", as_index=False)
      .agg(
          orgaos=("orgao", "count"),
          total_manifestacoes=("manifestacoes", "sum"),
          igro_medio_pct=("igro_pct", "mean"),
          pesquisas=("pesquisas", "sum")
      )
      .sort_values("classe_operacional")
)
tabela_classes["igro_medio_pct"] = tabela_classes["igro_medio_pct"].round(1)
display(tabela_classes)
save_table(tabela_classes, "tabela_4_distribuicao_operacional.csv")

## 3. Graficos centrais do artigo

Abaixo estao as sugestoes que espelham as chamadas do texto: Grafico 1 a Grafico 6, alem do heatmap dos KRIs.

In [ ]:
# Grafico 1 - distribuicao de manifestacoes por classe operacional
fig, ax1 = plt.subplots(figsize=(14, 7))
order = tabela_classes["classe_operacional"]
sns.barplot(data=tabela_classes, x="classe_operacional", y="total_manifestacoes", color="#2D6A4F", ax=ax1)
ax1.set_title("Grafico 1 - Distribuicao de manifestacoes por classe operacional")
ax1.set_xlabel("Classe operacional")
ax1.set_ylabel("Manifestacoes")

ax2 = ax1.twinx()
ax2.plot(range(len(tabela_classes)), tabela_classes["igro_medio_pct"], color="#C1121F", marker="o", linewidth=2.5)
ax2.set_ylabel("IGRO medio (%)")
ax2.set_ylim(0, 100)
save_fig(fig, "grafico_1_distribuicao_por_classe.png")
plt.show()

# Grafico 2 - TMR por classe operacional
fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=df, x="classe_operacional", y="tmr_dias", color="#669BBC", ax=ax)
sns.stripplot(data=df, x="classe_operacional", y="tmr_dias", color="#003049", alpha=0.55, size=6, ax=ax)
ax.axhline(10, color="#C1121F", linestyle="--", linewidth=2, label="Goalpost inferior TMR = 10 dias")
ax.axhline(5, color="#2A9D8F", linestyle=":", linewidth=2, label="Meta de excelencia TMR = 5 dias")
ax.set_title("Grafico 2 - Distribuicao do TMR por classe operacional")
ax.set_xlabel("Classe operacional")
ax.set_ylabel("TMR (dias)")
ax.legend(loc="upper left")
save_fig(fig, "grafico_2_boxplot_tmr_por_classe.png")
plt.show()

# Grafico 3 - PMA por orgao
top_pma = df.sort_values("pma_pct", ascending=False)
fig, ax = plt.subplots(figsize=(14, 14))
colors = np.where(top_pma["pma_pct"] > 2, "#C1121F", "#2A9D8F")
ax.barh(top_pma["orgao"], top_pma["pma_pct"], color=colors)
ax.axvline(2, color="#1D3557", linestyle="--", linewidth=2, label="Limite aceitavel PMA = 2%")
ax.axvline(1, color="#457B9D", linestyle=":", linewidth=2, label="Meta de excelencia PMA = 1%")
ax.set_title("Grafico 3 - Percentual de manifestacoes em atraso (PMA) por orgao")
ax.set_xlabel("PMA (%)")
ax.set_ylabel("Orgao")
ax.legend(loc="lower right")
save_fig(fig, "grafico_3_pma_por_orgao.png")
plt.show()

In [ ]:
# Grafico 4 - comparacao entre indicadores de qualidade
quality_long = pd.DataFrame(
    {
        "indicador": np.repeat(["RP (%)", "%RI (%)", "NR (NPS)"], len(df)),
        "valor": np.concatenate([df["rp_pct"].to_numpy(), df["ri_pct"].to_numpy(), df["nr_nps"].to_numpy()])
    }
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.boxplot(data=df, y="rp_pct", color="#2A9D8F", ax=axes[0])
axes[0].set_title("RP (%)")
axes[0].set_ylabel("Percentual")

sns.boxplot(data=df, y="ri_pct", color="#E76F51", ax=axes[1])
axes[1].set_title("%RI (%)")
axes[1].set_ylabel("Percentual")

sns.boxplot(data=df, y="nr_nps", color="#264653", ax=axes[2])
axes[2].set_title("NR (NPS)")
axes[2].set_ylabel("NPS")

fig.suptitle("Grafico 4 - Comparacao entre indicadores de qualidade", y=1.02)
fig.tight_layout()
save_fig(fig, "grafico_4_comparacao_indicadores_qualidade.png")
plt.show()

# Grafico 5 - correlacao entre RP e NR
fig, ax = plt.subplots(figsize=(10, 8))
sns.regplot(data=df, x="rp_pct", y="nr_nps", scatter=False, color="#1D3557", ax=ax)
scatter = ax.scatter(df["rp_pct"], df["nr_nps"], s=np.maximum(df["pesquisas"], 10) * 1.5, c=df["classe_operacional"].astype(int), cmap="viridis", alpha=0.75)
ax.set_title("Grafico 5 - Correlacao entre RP e NR")
ax.set_xlabel("Resolutividade percebida - RP (%)")
ax.set_ylabel("Nota de recomendacao - NR (NPS)")
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Classe operacional")

if spearmanr is not None:
    coef, p_value = spearmanr(df["rp_pct"], df["nr_nps"])
    ax.text(0.02, 0.98, f"Spearman = {coef:.3f}\np-valor = {p_value:.4g}", transform=ax.transAxes, ha="left", va="top", bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "#999"})

save_fig(fig, "grafico_5_correlacao_rp_nr.png")
plt.show()

# Grafico 6 - distribuicao do IGRO por orgao com faixas de risco
fig, ax = plt.subplots(figsize=(16, 14))
plot_df = df.sort_values("igro_pct", ascending=True)
palette = {"Controlado": "#2A9D8F", "Atencao": "#E9C46A", "Elevado": "#F4A261", "Critico": "#E63946"}
ax.barh(plot_df["orgao"], plot_df["igro_pct"], color=plot_df["faixa_risco"].map(palette))
for threshold in [50, 70, 90]:
    ax.axvline(threshold, color="#6C757D", linestyle="--", linewidth=1.5)
ax.set_title("Grafico 6 - Distribuicao do IGRO por orgao")
ax.set_xlabel("IGRO (%)")
ax.set_ylabel("Orgao")
save_fig(fig, "grafico_6_distribuicao_igro_por_orgao.png")
plt.show()

In [ ]:
# Figura 4 - heatmap dos KRIs por orgao
heatmap_cols = [
    "score_tmr_pct",
    "score_rdp_pct",
    "score_tr_pct",
    "score_ri_pct",
    "score_nr_pct",
    "igro_pct"
]
heatmap_df = df.set_index("orgao")[heatmap_cols].sort_values("igro_pct", ascending=False)
heatmap_df = heatmap_df.rename(columns={
    "score_tmr_pct": "Score TMR",
    "score_rdp_pct": "Score PMA",
    "score_tr_pct": "Score RP",
    "score_ri_pct": "Score %RI",
    "score_nr_pct": "Score NR",
    "igro_pct": "IGRO"
})

fig, ax = plt.subplots(figsize=(12, 16))
sns.heatmap(heatmap_df, cmap="RdYlGn", vmin=0, vmax=100, linewidths=0.2, linecolor="white", ax=ax)
ax.set_title("Figura 4 - Heatmap dos KRIs e do IGRO por orgao")
ax.set_xlabel("Indicadores")
ax.set_ylabel("Orgao")
save_fig(fig, "figura_4_heatmap_kri_por_orgao.png")
plt.show()

## 4. Tabelas auxiliares para a narrativa do artigo

Estas saidas ajudam a preencher a secao de casos extremos, checar top/bottom e apoiar a redacao da discussao.

In [ ]:
colunas_quadro = [
    "orgao", "classe_operacional", "manifestacoes", "pesquisas", "tmr_dias", "pma_pct", "rp_pct", "ri_pct", "nr_nps", "igro_pct", "faixa_risco", "amostra_insuficiente"
]
top10 = df.nlargest(10, "igro_pct")[colunas_quadro]
bottom10 = df.nsmallest(10, "igro_pct")[colunas_quadro]
quadro_extremos = pd.concat([
    top10.assign(grupo="Top 10"),
    bottom10.assign(grupo="Bottom 10")
], ignore_index=True)
display(quadro_extremos)
save_table(quadro_extremos, "quadro_extremos_top_bottom_10.csv")

comparacao_extremos = pd.concat([
    df.nlargest(1, "igro_pct")[colunas_quadro].assign(perfil="Excelencia"),
    df.nsmallest(1, "igro_pct")[colunas_quadro].assign(perfil="Critico")
], ignore_index=True)
display(comparacao_extremos)
save_table(comparacao_extremos, "quadro_1_comparacao_extremos.csv")

## 5. Analises extras recomendadas

Aqui entram sugestoes que nao aparecem de forma explicita na lista do artigo, mas tendem a melhorar a robustez da narrativa e a transparencia metodologica.

In [ ]:
# Extra A - sensibilidade de ranking com cenarios de ponderacao
def geometric_weighted_mean(frame: pd.DataFrame, columns: list[str], weights: list[float]) -> pd.Series:
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    safe = frame[columns].clip(lower=1e-6)
    return np.exp(np.log(safe).mul(weights, axis=1).sum(axis=1))

score_base_cols = ["score_tmr", "score_rdp", "score_tr", "score_ri", "score_nr"]
cenarios = {
    "uniforme_geometrico": [0.2, 0.2, 0.2, 0.2, 0.2],
    "tempestividade_pesada": [0.3, 0.3, 0.1333, 0.1333, 0.1334],
    "qualidade_pesada": [0.1333, 0.1333, 0.2445, 0.2445, 0.2444]
}

sens = df[["orgao", *score_base_cols]].copy()
for nome, pesos in cenarios.items():
    sens[nome] = geometric_weighted_mean(sens, score_base_cols, pesos)

sens["aritmetico_uniforme"] = sens[score_base_cols].mean(axis=1)

ranking_cols = ["uniforme_geometrico", "tempestividade_pesada", "qualidade_pesada", "aritmetico_uniforme"]
for col in ranking_cols:
    sens[f"rank_{col}"] = sens[col].rank(method="min", ascending=False)

display(sens[["orgao", *ranking_cols, *(f"rank_{c}" for c in ranking_cols)]].sort_values("rank_uniforme_geometrico").head(15))
save_table(sens[["orgao", *ranking_cols, *(f"rank_{c}" for c in ranking_cols)]], "sensibilidade_cenarios_ranking.csv")

if spearmanr is not None:
    base_rank = sens["rank_uniforme_geometrico"]
    corr_rows = []
    for col in ["rank_tempestividade_pesada", "rank_qualidade_pesada", "rank_aritmetico_uniforme"]:
        coef, p_value = spearmanr(base_rank, sens[col])
        corr_rows.append({"cenario": col, "spearman": coef, "p_valor": p_value})
    display(pd.DataFrame(corr_rows))

# Extra B - confiabilidade da parte perceptiva versus tamanho amostral
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(df["pesquisas"], df["nr_nps"], s=np.maximum(df["manifestacoes"], 10) / 15, alpha=0.7, color="#6A4C93")
ax.axvline(30, color="#6C757D", linestyle="--", linewidth=2, label="30 pesquisas")
ax.set_title("Extra - NPS por tamanho da amostra de pesquisas")
ax.set_xlabel("Numero de pesquisas")
ax.set_ylabel("NR (NPS)")
ax.legend()
save_fig(fig, "extra_nps_vs_tamanho_amostra.png")
plt.show()

## 6. Proximos passos sugeridos

Sugestoes de uso depois de rodar o notebook:
- decidir se o artigo vai trabalhar com `51` ou `52` orgaos e documentar a regra de exclusao, se houver;
- revisar se `rdp` realmente corresponde ao conceito editorial de `PMA` antes da versao final;
- aproveitar a secao de sensibilidade para preencher a subsecao 4.6 do artigo;
- usar as tabelas exportadas em `09_resultados/artigo_igro_figuras_tabelas/` como base para quadros e anexos;
- considerar um grafico extra de confiabilidade por amostra quando for discutir limitacoes dos indicadores perceptivos.